# 随机追加验证与耗时审计

## tl;dr

本轮随机补抽 20260320、20260401、20260616，新增恢复／对比的同进程分项计时。与已有日期合计 7 日、14 个全市场任务。任务尚未全部结束，当前输出不是最终全量通过结论。最终结果由 campaign.json / campaign.md 持续更新。

## Context & Methods

候选日期要求 SH 逐笔、SH raw snapshot、SZ 委托、SZ 成交、SZ raw snapshot 五个 Parquet 文件齐全；排除已选 20260828、20260601、20260706、20260806 后，以种子 20260907 无放回抽取 3 日。股票和 ETF 全市场，样本不计入。

### Key Assumptions

- 旧批次保留冻结的非埋点二进制；新增批次使用可选 profiling benchmark，不改市场适配或比较规则。沪深合成报告逐字段等价检查已通过。
- 单次调用中，恢复 = 输入分片 + 回放循环扣除验证回调 + 清理；对比 = 参考加载 + 验证回调 + 报告整理；序列化写出另列。
- 时间都是 wall time，包含系统调度等待。恢复不是纯 apply CPU 时间；验证回调含候选状态提取。计时器开销不假装为零。
- 旧任务没有埋点则分项为空，不能通过两个不同运行耗时相减伪造拆分。
- 每日任务跨度是最早启动到最晚结束，可能含错峰等待；各市场分项相加是累计工作量，不是并行后的日历耗时。
- 匹配率以可比帧为分母，状态排除、错误、缺少来源分列。中止或未完成不等于零差异通过。

## Data

### 1. 来源、随机选择与二进制证据

仅需 Python 标准库；本 notebook 不扫描全市场原始数据，不重跑验证。

In [1]:
from pathlib import Path
import json, random, hashlib, runpy
ROOT = Path.cwd()
if not (ROOT / 'Cargo.toml').exists(): ROOT = ROOT.parent
OUT = ROOT / 'reports/20260906-random-profiled-full'
manifest = json.loads((OUT / 'manifest.json').read_text())
assert sorted(random.Random(manifest['seed']).sample(manifest['eligible_dates'], 3)) == manifest['days']
assert not set(manifest['days']) & set(manifest['excluded_dates'])
assert hashlib.sha256(Path(manifest['frozen_binary']).read_bytes()).hexdigest() == manifest['binary_sha256']
for name, expected in json.loads((OUT / 'source-sha256.json').read_text()).items():
    assert hashlib.sha256((OUT / 'source-snapshot' / name).read_bytes()).hexdigest() == expected
print('seed', manifest['seed'], 'pool', len(manifest['eligible_dates']), 'selected', manifest['days'])
print('binary SHA256', manifest['binary_sha256'])
for day in manifest['days']:
    print(day, {s['feed']: s['rows'] for s in manifest['sources'] if s['date'] == day})


seed 20260907 pool 155 selected ['20260320', '20260401', '20260616']
binary SHA256 bdb1a4d58a0bfee30cae7d5582b87d98b4db8fbe93bf023f81a7131159548a9d
20260320 {'mdl_4_24_0': 205994077, 'MarketData': 16937817, 'mdl_6_33_0': 161176742, 'mdl_6_36_0': 147550833, 'mdl_6_28_0': 15922752}
20260401 {'mdl_4_24_0': 187236810, 'MarketData': 16724655, 'mdl_6_33_0': 146628662, 'mdl_6_36_0': 132858624, 'mdl_6_28_0': 15455882}
20260616 {'mdl_4_24_0': 231793674, 'MarketData': 17221949, 'mdl_6_33_0': 179654484, 'mdl_6_36_0': 163570047, 'mdl_6_28_0': 15797575}


## Results

### 2. 全量覆盖和匹配结果

REQUIRE_COMPLETE=True 可启用最终完整性门槛。未完成阶段保留明确状态，不使用全零计数冒充完成。

In [2]:
collect = runpy.run_path(str(ROOT / 'analysis/summarize_validation_campaign.py'))['collect']
result = collect(OUT)
REQUIRE_COMPLETE = False
if REQUIRE_COMPLETE: assert result['all_jobs_finished'], 'Campaign incomplete'
assert len(result['rows']) == 14
assert len({(r['date'], r['market']) for r in result['rows']}) == 14
print('as of', result['updated_at'], 'completed', result['completed_market_jobs'], '/', result['expected_market_jobs'])
print('all finished', result['all_jobs_finished'], 'all passed', result['all_snapshot_validations_passed'])
for row in result['rows']:
    print({k: row[k] for k in ('date','market','status','matched','mismatched','excluded_by_status','data_errors','missing_source','elapsed_seconds') if k in row})


as of 2026-09-06T15:41:46.544458+00:00 completed 2 / 14
all finished False all passed False
{'date': '20260320', 'market': 'SH', 'status': 'running'}
{'date': '20260320', 'market': 'SZ', 'status': 'running'}
{'date': '20260401', 'market': 'SH', 'status': 'queued'}
{'date': '20260401', 'market': 'SZ', 'status': 'queued'}
{'date': '20260601', 'market': 'SH', 'status': 'running'}
{'date': '20260601', 'market': 'SZ', 'status': 'running'}
{'date': '20260616', 'market': 'SH', 'status': 'queued'}
{'date': '20260616', 'market': 'SZ', 'status': 'queued'}
{'date': '20260706', 'market': 'SH', 'status': 'queued'}
{'date': '20260706', 'market': 'SZ', 'status': 'queued'}
{'date': '20260806', 'market': 'SH', 'status': 'queued'}
{'date': '20260806', 'market': 'SZ', 'status': 'queued'}
{'date': '20260828', 'market': 'SH', 'status': 'completed', 'matched': 12342471, 'mismatched': 0, 'excluded_by_status': 5, 'data_errors': 0, 'missing_source': 0, 'elapsed_seconds': 5394.292}
{'date': '20260828', 'market'

### 3. 时间分拆及每日加总

记录秒数，不对旧批次补造缺失的计时。

In [3]:
for row in result['rows']:
    if 'timings' not in row:
        print(row['date'], row['market'], 'no completed phase timings', row['status'])
        continue
    stages = row['timings']['stages']
    assert all(stages[k] >= 0 for k in stages if k.endswith('_seconds'))
    assert abs(stages['profiled_total_seconds'] - stages['restore_total_seconds'] - stages['validation_total_seconds'] - stages['unattributed_seconds']) < 1e-6
    print(row['date'], row['market'], stages)
for day in result['daily']:
    print(day)


20260320 SH no completed phase timings running
20260320 SZ no completed phase timings running
20260401 SH no completed phase timings queued
20260401 SZ no completed phase timings queued
20260601 SH no completed phase timings running
20260601 SZ no completed phase timings running
20260616 SH no completed phase timings queued
20260616 SZ no completed phase timings queued
20260706 SH no completed phase timings queued
20260706 SZ no completed phase timings queued
20260806 SH no completed phase timings queued
20260806 SZ no completed phase timings queued
20260828 SH no completed phase timings completed
20260828 SZ no completed phase timings completed
{'date': '20260320', 'complete': False, 'matched': 0, 'mismatched': 0}
{'date': '20260401', 'complete': False, 'matched': 0, 'mismatched': 0}
{'date': '20260601', 'complete': False, 'matched': 0, 'mismatched': 0}
{'date': '20260616', 'complete': False, 'matched': 0, 'mismatched': 0}
{'date': '20260706', 'complete': False, 'matched': 0, 'mismatc

### 4. 异常和证据边界

失败明细最多 5000 条，聚合计数完整；首批异常只用于定位，不代替总体分布。

In [4]:
for row in result['rows']:
    if row.get('error'): print(row['date'], row['market'], row['error'][-1500:])
    if 'mismatch_fields' in row:
        print(row['date'], row['market'], 'mismatch fields', row['mismatch_fields'], 'noncomparable', row['not_comparable_reasons'])
    for anomaly in row.get('sample_anomalies', []): print(anomaly)


20260828 SH mismatch fields {} noncomparable {'excluded phase status SUSP': 5}
20260828 SZ mismatch fields {} noncomparable {'excluded phase status C1': 4, 'excluded phase status H0': 6, 'excluded phase status S1': 4, 'excluded phase status T1': 4}


## Takeaways

当前分析只验证已产生的报告及计时归属，不能把未结束的 12 个市场任务算作通过。结束后的权威汇总是 reports/20260906-random-profiled-full/campaign.json；重新执行本 notebook 可核对最终状态。

同一进程中的排他性计时优于跨运行相减，但埋点开销和共享资源竞争仍影响结果。不同日期或并发负载不能直接当成受控性能对照；快照全匹配也不证明每个逐事件中间态和 FIFO 都正确。